In [ ]:
# JSON Generator (CARRO-aware) v2 — input dinâmico (corrigido)
# ------------------------------------------------------------

import json
from pathlib import Path
import pandas as pd

# ===== Auxiliar: detectar se estamos no Colab =====
IN_COLAB = False
try:
    import google.colab  # type: ignore
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ===== Obter arquivo de entrada (dinâmico) =====
if IN_COLAB:
    print("⤴️ Selecione seu arquivo Excel (.xlsx) para upload...")
    uploaded = files.upload()  # retorna dict {filename: bytes}
    if not uploaded:
        raise RuntimeError("Nenhum arquivo foi enviado.")
    input_name = list(uploaded.keys())[0]
    input_path = Path(input_name)
    print(f"✔️ Arquivo recebido: {input_name}")
else:
    input_name = input("Informe o caminho do arquivo Excel (.xlsx): ").strip()
    if not input_name:
        raise RuntimeError("Caminho do arquivo não informado.")
    input_path = Path(input_name)
    if not input_path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {input_path}")

# ===== Nome do JSON de saída =====
output_path = input_path.with_name(input_path.stem + ".json")
print(f"📝 Saída JSON será: {output_path}")

# ===== Funções utilitárias =====
def norm_text(s):
    return (str(s).strip() if pd.notna(s) else "")

def first_non_empty(*vals):
    for v in vals:
        if pd.notna(v) and str(v).strip() != "":
            return str(v).strip()
    return ""

def slug(s):
    import re
    s = (s or "").lower()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^a-z0-9_\-]", "", s)
    return s

# ===== Carregar planilha e escolher a aba útil =====
xls = pd.ExcelFile(input_path, engine="openpyxl")

preferred_cols = {
    "Item": ["Item", "item", "Código", "Cod", "A"],
    "Local": ["Local", "local"],
    "Sistema": ["Sistema", "sistema"],
    "Subsistema": ["Subsistema", "subsistema"],
}

chosen_sheet = None
for sh in xls.sheet_names:
    try:
        temp_df = pd.read_excel(xls, sheet_name=sh, engine="openpyxl")
        temp_df.columns = [str(c).strip() for c in temp_df.columns]
        has_item = any(a in temp_df.columns for a in preferred_cols["Item"])
        has_sistema = any(a in temp_df.columns for a in preferred_cols["Sistema"])
        has_carro = any(str(c).strip().lower().startswith("carro") for c in temp_df.columns)
        if (has_item and has_sistema) or has_carro:
            chosen_sheet = sh
            break
    except Exception:
        pass

if chosen_sheet is None:
    chosen_sheet = xls.sheet_names[0]
    print(f"ℹ️ Nenhuma aba 'preferida' encontrada; usando a primeira: {chosen_sheet}")
else:
    print(f"✔️ Aba selecionada: {chosen_sheet}")

# ===== Ler a aba escolhida =====
df = pd.read_excel(xls, sheet_name=chosen_sheet, engine="openpyxl")
df.columns = [str(c).strip() for c in df.columns]

# ===== Construir mapa de colunas =====
col_map = {
    "Item": ["Item", "item", "Código", "Cod", "A"],
    "Local": ["Local", "local"],
    "Sistema": ["Sistema", "sistema"],
    "Subsistema": ["Subsistema", "subsistema"],
    "TIPO ATIVIDADE": ["TIPO ATIVIDADE", "Tipo", "Tipo Atividade"],
    "DESCRIÇÃO CURTA": ["DESCRIÇÃO CURTA", "Descricao Curta", "Descrição Curta"],
    "DESCRIÇÃO LONGA": ["DESCRIÇÃO LONGA", "Descricao Longa", "Descrição Longa"],
    "REFERÊNCIA": ["REFERÊNCIA", "Referencia", "Referência"],
    "HYPERLINK REFERÊNCIA": ["HYPERLINK REFERÊNCIA", "Hyperlink Referência", "Link Referência", "HYPERLINK"],
    "ACESSO": ["ACESSO", "Acesso"],
    "FERRAMENTA": ["FERRAMENTA", "Ferramenta"],
    "MATERIAL": ["MATERIAL", "Material"],
}

resolved = {}
for key, aliases in col_map.items():
    found = None
    for a in aliases:
        if a in df.columns:
            found = a
            break
    resolved[key] = found

# ===== Detectar colunas CARRO =====
carro_cols = [c for c in df.columns if str(c).strip().lower().startswith("carro")]
print(f"🚘 Colunas CARRO detectadas: {carro_cols}")

# ===== Montar JSON =====
sections = {}

for _, row in df.iterrows():
    item_code = norm_text(row[resolved["Item"]]) if resolved["Item"] else ""
    if item_code == "":
        continue

    local = norm_text(row[resolved["Local"]]) if resolved["Local"] else ""
    sistema = norm_text(row[resolved["Sistema"]]) if resolved["Sistema"] else ""
    subsistema = norm_text(row[resolved["Subsistema"]]) if resolved["Subsistema"] else ""
    tipo = norm_text(row[resolved["TIPO ATIVIDADE"]]) if resolved["TIPO ATIVIDADE"] else ""
    desc_curta = norm_text(row[resolved["DESCRIÇÃO CURTA"]]) if resolved["DESCRIÇÃO CURTA"] else ""
    desc_longa = norm_text(row[resolved["DESCRIÇÃO LONGA"]]) if resolved["DESCRIÇÃO LONGA"] else ""
    ref_code = norm_text(row[resolved["REFERÊNCIA"]]) if resolved["REFERÊNCIA"] else ""
    ref_url = norm_text(row[resolved["HYPERLINK REFERÊNCIA"]]) if resolved["HYPERLINK REFERÊNCIA"] else ""
    acesso = norm_text(row[resolved["ACESSO"]]) if resolved["ACESSO"] else ""
    ferramenta = norm_text(row[resolved["FERRAMENTA"]]) if resolved["FERRAMENTA"] else ""
    material = norm_text(row[resolved["MATERIAL"]]) if resolved["MATERIAL"] else ""

    carros = {}
    for c in carro_cols:
        val = row[c]
        carros[c] = (str(val).strip().upper() == "X") if pd.notna(val) else False

    item_obj = {
        "id": slug(f"{item_code}_{sistema}_{subsistema}"),
        "item_code": item_code,
        "label": desc_curta,
        "descricao_longa": desc_longa,
        "referencia_code": ref_code,
        "referencia_url": ref_url,
        "helper": first_non_empty(acesso, ferramenta, material),
        "local": local,
        "sistema": sistema,
        "subsistema": subsistema,
        "tipo_atividade": tipo,
        "acesso": acesso,
        "ferramenta": ferramenta,
        "material": material,
        "carros": carros,
    }

    sec_key = (sistema, subsistema)
    if sec_key not in sections:
        sec_name = f"{sistema} - {subsistema}" if subsistema else (sistema or "Seção")
        sections[sec_key] = {"name": sec_name, "items": []}
    sections[sec_key]["items"].append(item_obj)

schema = {
    "title": f"{input_path.stem}",
    "carro_columns": carro_cols,
    "sections": list(sections.values()),
}

# ===== Grava o JSON =====
output_path.write_text(
    json.dumps(schema, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(f"✅ JSON gerado: {output_path}")

# ===== Download automático (somente no Colab) =====
if IN_COLAB:
    files.download(str(output_path))


⤴️ Selecione seu arquivo Excel (.xlsx) para upload...


Saving ON_1MRO - SÉRIE 7000 - MANUTENÇÃO PREVENTIVA N1 (12.500 km).xlsx to ON_1MRO - SÉRIE 7000 - MANUTENÇÃO PREVENTIVA N1 (12.500 km).xlsx
✔️ Arquivo recebido: ON_1MRO - SÉRIE 7000 - MANUTENÇÃO PREVENTIVA N1 (12.500 km).xlsx
📝 Saída JSON será: ON_1MRO - SÉRIE 7000 - MANUTENÇÃO PREVENTIVA N1 (12.500 km).json
✔️ Aba selecionada: Planilha1
🚘 Colunas CARRO detectadas: ['CARRO 1', 'CARRO 2', 'CARRO 3', 'CARRO 4', 'CARRO 5', 'CARRO 6', 'CARRO 7', 'CARRO 8']
✅ JSON gerado: ON_1MRO - SÉRIE 7000 - MANUTENÇÃO PREVENTIVA N1 (12.500 km).json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>